# N-grams Playground — coding workshop (advanced)

**Goes with the Session 1 lesson, Step 1 · N-grams.** Expected time: ~20 minutes,
individual. No GPU needed.

In the lesson you built a bigram model from three sentences, in the browser, and saw the
sparsity problem: a pair the corpus never saw gets probability 0. The lesson also claimed
that a Galician model has fewer sentences to count from than a Spanish one, so its
estimates are noisier. Here you measure both, on real text:

- **Part A:** how much Galician and Spanish text exists on Wikipedia.
- **Part B:** how many bigrams of new text the model has never seen, as the training text
  grows from 10 thousand to 1.8 million words, in both languages.
- **Part C:** what a longer n does to sparsity.
- **Part D:** a Galician model writes text.

## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried, or run the
   cell under it: that puts the solution into a new code cell for you, ready to run.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

## Part A — How much text is there?

Wikipedia is one of the largest freely licensed sources of text in both languages. The
course counted every word in the November 2023 Wikipedia dumps of both languages (the
`wikimedia/wikipedia` dataset on Hugging Face; a word is a run of letters):

| Wikipedia, November 2023 | articles | words |
|---|---|---|
| Galician | 200,092 | 70.9 million |
| Spanish | 1,841,155 | 891.2 million |
| Spanish ÷ Galician | 9.2 | 12.6 |

The full dumps are 0.3 GB and 3.5 GB to download, so this notebook uses a sample: about 2
million words of each, from randomly chosen articles. The samples are on the course site,
under the Wikipedia license (CC BY-SA 4.0).

In [ ]:
import gzip
import re
import time
import urllib.request
from collections import Counter

import matplotlib.pyplot as plt

DATA_URL = "https://olzama.github.io/teaching/GreenAI/corpora"
WORD = re.compile(r"[^\W\d_]+")


def load_articles(lang):
    """Download the 2-million-word sample for `lang` ("gl" or "es"): one list of words per article."""
    path = f"wiki_{lang}_2M.txt.gz"
    urllib.request.urlretrieve(f"{DATA_URL}/{path}", path)
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [WORD.findall(line.lower()) for line in f]


articles = {lang: load_articles(lang) for lang in ["gl", "es"]}
for lang, arts in articles.items():
    print(f"{lang}: {len(arts):,} articles, {sum(map(len, arts)):,} words; "
          f"first words: {' '.join(arts[0][:12])}")

### Training text and test text

The last 200,000 words of each sample are the *test text*: new text the model never
counts. The rest is the *training text*, from which we take the first N words.

In [ ]:
def split_off_test(arts, test_words=200_000):
    """Move whole articles from the end of `arts` into a test set of at least `test_words` words."""
    arts = list(arts)
    test, n = [], 0
    while n < test_words:
        test.append(arts.pop())
        n += len(test[-1])
    return arts, test


def first_words(arts, n_words):
    """The first n_words words of the training text, still split into articles."""
    out, n = [], 0
    for a in arts:
        if n >= n_words:
            break
        out.append(a[:n_words - n])
        n += len(out[-1])
    return out


train, test = {}, {}
for lang in articles:
    train[lang], test[lang] = split_off_test(articles[lang])
    print(f"{lang}: {sum(map(len, train[lang])):,} training words, {sum(map(len, test[lang])):,} test words")

## Part B — Unseen bigrams

`count_ngrams` counts every n-gram (n words in a row) inside each article. With n = 2
these are the counts of the lesson's bigram model, on real text.

### ✏️ Exercise 1 — Count n-grams, and the share never seen

Complete `count_ngrams` and `unseen_rate`.

- `count_ngrams` counts every n-gram (a tuple of n words in a row) inside each article.
  `zip(words, words[1:])` gives the bigrams of `words`; the general form zips n shifted
  copies of the list.
- `unseen_rate` is the share of the test n-grams, counted with repetition, that never
  occur in the training counts.

The cell should print 0.6666…: two of the three bigrams of “my uncle stayed quiet” are
new.

<details><summary><b>Hints</b> (click to open)</summary>

- `words[i:]` for i = 0, …, n − 1 are n shifted copies; `zip(*...)` walks them together.
- A `Counter` updated with an iterable of tuples adds 1 per tuple.
- `unseen_rate`: the condition is `gram not in train_counts`; divide by the number of test n-grams, `sum(test_counts.values())`.

</details>

In [ ]:
def count_ngrams(arts, n):
    """Counter of n-grams (tuples of n words), counted inside each article."""
    counts = Counter()
    for words in arts:
        # TODO: add the n-grams of `words` to `counts` (zip n shifted copies of the list)
        ___
    return counts


def unseen_rate(train_counts, test_counts):
    """Share of the n-grams in the test text that never occur in the training text."""
    # TODO: the share of test n-grams (with repetition) missing from train_counts
    ___


bigrams = count_ngrams([["my", "aunt", "stayed", "quiet"], ["my", "aunt", "spoke", "wisely"]], 2)
print(bigrams)
print(unseen_rate(bigrams, count_ngrams([["my", "uncle", "stayed", "quiet"]], 2)))

In [ ]:
# Check for Exercise 1. Run this cell as is.
def test_bigrams():
    """bigrams of the lesson's corpus"""
    c = count_ngrams([["my", "aunt", "stayed", "quiet"], ["my", "aunt", "spoke", "wisely"]], 2)
    assert c[("my", "aunt")] == 2, "'my aunt' occurs twice"
    assert sum(c.values()) == 6, "two sentences of 4 words have 3 bigrams each"

def test_inside_articles():
    """no n-gram crosses from one article into the next"""
    c = count_ngrams([["a", "b"], ["c", "d"]], 2)
    assert ("b", "c") not in c, "'b c' spans two articles"

def test_trigrams_and_short():
    """trigrams, and an article shorter than n"""
    c = count_ngrams([["a", "b", "c", "d"], ["x"]], 3)
    assert c == {("a", "b", "c"): 1, ("b", "c", "d"): 1}

def test_unigrams():
    """n = 1 counts words, as 1-tuples"""
    assert count_ngrams([["a", "a", "b"]], 1) == {("a",): 2, ("b",): 1}

def test_unseen_rate():
    """unseen rate counts repeated test n-grams every time"""
    train_c = count_ngrams([["a", "b"]], 2)
    assert abs(unseen_rate(train_c, count_ngrams([["my", "uncle", "stayed", "quiet"]], 2)) - 1) < 1e-12
    assert abs(unseen_rate(train_c, count_ngrams([["a", "b", "a", "b", "x"]], 2)) - 2 / 4) < 1e-12, \
        "'a b' twice is seen, 'b a' and 'b x' are not: 2 of 4"

_failed = 0
for _test in [test_bigrams, test_inside_articles, test_trigrams_and_short, test_unigrams, test_unseen_rate]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def count_ngrams(arts, n):
    """Counter of n-grams (tuples of n words), counted inside each article."""
    counts = Counter()
    for words in arts:
        counts.update(zip(*(words[i:] for i in range(n))))
    return counts


def unseen_rate(train_counts, test_counts):
    """Share of the n-grams in the test text that never occur in the training text."""
    unseen = sum(c for gram, c in test_counts.items() if gram not in train_counts)
    return unseen / sum(test_counts.values())


bigrams = count_ngrams([["my", "aunt", "stayed", "quiet"], ["my", "aunt", "spoke", "wisely"]], 2)
print(bigrams)
print(unseen_rate(bigrams, count_ngrams([["my", "uncle", "stayed", "quiet"]], 2)))
```

</details>

In [ ]:
# Stuck on Exercise 1? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 1" above; your own cell stays as it is.)
get_ipython().set_next_input("def count_ngrams(arts, n):\n    \"\"\"Counter of n-grams (tuples of n words), counted inside each article.\"\"\"\n    counts = Counter()\n    for words in arts:\n        counts.update(zip(*(words[i:] for i in range(n))))\n    return counts\n\n\ndef unseen_rate(train_counts, test_counts):\n    \"\"\"Share of the n-grams in the test text that never occur in the training text.\"\"\"\n    unseen = sum(c for gram, c in test_counts.items() if gram not in train_counts)\n    return unseen / sum(test_counts.values())\n\n\nbigrams = count_ngrams([[\"my\", \"aunt\", \"stayed\", \"quiet\"], [\"my\", \"aunt\", \"spoke\", \"wisely\"]], 2)\nprint(bigrams)\nprint(unseen_rate(bigrams, count_ngrams([[\"my\", \"uncle\", \"stayed\", \"quiet\"]], 2)))")


The check above uses the lesson's corpus: “my uncle stayed quiet” has three bigrams, and
two of them (“my uncle”, “uncle stayed”) never occur in the first two sentences, so the
unseen rate is 2/3.

Now the real text.

In [ ]:
SIZES = [10_000, 30_000, 100_000, 300_000, 1_000_000, 1_800_000]

t0 = time.perf_counter()

unseen = {lang: [] for lang in train}
for lang in train:
    test_bigrams = count_ngrams(test[lang], 2)
    for n_words in SIZES:
        train_bigrams = count_ngrams(first_words(train[lang], n_words), 2)
        unseen[lang].append(unseen_rate(train_bigrams, test_bigrams))

seconds = time.perf_counter() - t0
print(f"The whole sweep: {seconds:.1f} s\n")

print(f"{'training words':>15} | {'Galician':>8} | {'Spanish':>8}")
for i, n_words in enumerate(SIZES):
    print(f"{n_words:>15,} | {unseen['gl'][i]:>8.1%} | {unseen['es'][i]:>8.1%}")

In [ ]:
plt.figure(figsize=(6.5, 4))
for lang, name in [("gl", "Galician"), ("es", "Spanish")]:
    plt.plot(SIZES, [100 * u for u in unseen[lang]], "o-", label=name)
plt.xscale("log")
plt.xlabel("training words (log scale)")
plt.ylabel("test bigrams never seen in training, %")
plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

### ✏️ Exercise 2 — Smoothed probabilities and perplexity (advanced)

The unseen rate counts zeros. *Perplexity* measures how surprised a smoothed model
is by the test text: with N test bigrams,

PP = exp( −(1/N) · Σ log P(w₂ | w₁) ),   P(w₂ | w₁) = (count(w₁ w₂) + k) / (count(w₁) + k·V)

where V is the size of the vocabulary and k = 0.1 (add-k smoothing, the lesson's slider).
Lower is better. Keep V fixed for every training size: the number of different words in
the whole sample, training and test text together. (If V grew with the training text, a
bigger text would spread more smoothing over more words, and the scores would not be
comparable.) Compute the perplexity for Galician and Spanish at every training size.

<details><summary><b>Hints</b> (click to open)</summary>

- A `Counter` returns 0 for a missing key, so `big[(w1, w2)]` and `first[w1]` work for unseen words too.
- Each test bigram counts `c` times: multiply its log-probability by `c`.
- Return `math.exp(-log_sum / n)`.

</details>

In [ ]:
import math


def bigram_perplexity(train_arts, test_arts, V, k=0.1):
    big = count_ngrams(train_arts, 2)
    first = Counter()
    for (w1, _), c in big.items():
        first[w1] += c                      # count(w1) as the first word of a pair
    test_big = count_ngrams(test_arts, 2)
    # TODO: the sum over test bigrams (with repetition) of log P(w2 | w1), add-k smoothed
    log_sum = ___
    n = sum(test_big.values())
    return ___


V_all = {lang: len(count_ngrams(train[lang] + test[lang], 1)) for lang in train}
perplexity = {lang: [bigram_perplexity(first_words(train[lang], s), test[lang], V_all[lang]) for s in SIZES]
              for lang in train}
for i, s in enumerate(SIZES):
    print(f"{s:>10,} words: Galician {perplexity['gl'][i]:8.0f}   Spanish {perplexity['es'][i]:8.0f}")

In [ ]:
# Check for Exercise 2. Run this cell as is.
def test_hand_example():
    """a hand-computed example"""
    # train "a b", test "a b": V = 2, P(b | a) = (1 + 0.1) / (1 + 0.2)
    pp = bigram_perplexity([["a", "b"]], [["a", "b"]], 2)
    assert abs(pp - (1.2 / 1.1)) < 1e-9, f"expected {1.2 / 1.1:.4f}, got {pp:.4f}"

def test_unseen_is_finite():
    """smoothing keeps unseen bigrams finite"""
    pp = bigram_perplexity([["a", "b"]], [["b", "a"], ["x", "y"]], 4)
    assert math.isfinite(pp) and pp > 1

def test_more_text_helps():
    """more training text lowers perplexity"""
    for lang in perplexity:
        assert perplexity[lang][-1] < perplexity[lang][0]

_failed = 0
for _test in [test_hand_example, test_unseen_is_finite, test_more_text_helps]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
import math


def bigram_perplexity(train_arts, test_arts, V, k=0.1):
    big = count_ngrams(train_arts, 2)
    first = Counter()
    for (w1, _), c in big.items():
        first[w1] += c
    test_big = count_ngrams(test_arts, 2)
    log_sum = sum(c * math.log((big[(w1, w2)] + k) / (first[w1] + k * V))
                  for (w1, w2), c in test_big.items())
    n = sum(test_big.values())
    return math.exp(-log_sum / n)


V_all = {lang: len(count_ngrams(train[lang] + test[lang], 1)) for lang in train}
perplexity = {lang: [bigram_perplexity(first_words(train[lang], s), test[lang], V_all[lang]) for s in SIZES]
              for lang in train}
for i, s in enumerate(SIZES):
    print(f"{s:>10,} words: Galician {perplexity['gl'][i]:8.0f}   Spanish {perplexity['es'][i]:8.0f}")
```

</details>

In [ ]:
# Stuck on Exercise 2? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 2" above; your own cell stays as it is.)
get_ipython().set_next_input("import math\n\n\ndef bigram_perplexity(train_arts, test_arts, V, k=0.1):\n    big = count_ngrams(train_arts, 2)\n    first = Counter()\n    for (w1, _), c in big.items():\n        first[w1] += c\n    test_big = count_ngrams(test_arts, 2)\n    log_sum = sum(c * math.log((big[(w1, w2)] + k) / (first[w1] + k * V))\n                  for (w1, w2), c in test_big.items())\n    n = sum(test_big.values())\n    return math.exp(-log_sum / n)\n\n\nV_all = {lang: len(count_ngrams(train[lang] + test[lang], 1)) for lang in train}\nperplexity = {lang: [bigram_perplexity(first_words(train[lang], s), test[lang], V_all[lang]) for s in SIZES]\n              for lang in train}\nfor i, s in enumerate(SIZES):\n    print(f\"{s:>10,} words: Galician {perplexity['gl'][i]:8.0f}   Spanish {perplexity['es'][i]:8.0f}\")")


### ✏️ Exercise 3 — Prune the counts: memory against quality (advanced)

Large n-gram models do not store every n-gram: they drop the rare ones, which are
most of the table. Write `prune`, which keeps only the bigrams seen at least `min_count`
times. Then, on all the Galician training text, compare the number of stored bigrams with
the unseen rate (a pruned bigram counts as unseen).

<details><summary><b>Hints</b> (click to open)</summary>

- A dictionary comprehension over `counts.items()`, keeping pairs with `c >= min_count`.
- Wrap it in `Counter(...)` so the result is still a Counter.

</details>

In [ ]:
def prune(counts, min_count):
    # TODO: a Counter with only the n-grams seen at least min_count times
    return ___


full_gl = count_ngrams(train["gl"], 2)
test_big_gl = count_ngrams(test["gl"], 2)
pruned = {}
print(f"{'min count':>9} | {'stored bigrams':>14} | {'share of the full table':>23} | {'unseen rate':>11}")
for m in [1, 2, 3, 5]:
    pruned[m] = prune(full_gl, m)
    print(f"{m:>9} | {len(pruned[m]):>14,} | {len(pruned[m]) / len(full_gl):>23.0%} | "
          f"{unseen_rate(pruned[m], test_big_gl):>11.1%}")

In [ ]:
# Check for Exercise 3. Run this cell as is.
def test_prune_small():
    """keeps exactly the n-grams at or above min_count"""
    c = Counter({("a", "b"): 3, ("b", "c"): 1, ("c", "d"): 2})
    assert prune(c, 2) == Counter({("a", "b"): 3, ("c", "d"): 2})
    assert prune(c, 1) == c

def test_tradeoff():
    """pruning saves memory and costs quality"""
    assert len(pruned[2]) < len(pruned[1]) / 2, "most bigrams are seen once"
    assert unseen_rate(pruned[2], test_big_gl) > unseen_rate(pruned[1], test_big_gl)

_failed = 0
for _test in [test_prune_small, test_tradeoff]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
def prune(counts, min_count):
    return Counter({g: c for g, c in counts.items() if c >= min_count})


full_gl = count_ngrams(train["gl"], 2)
test_big_gl = count_ngrams(test["gl"], 2)
pruned = {}
print(f"{'min count':>9} | {'stored bigrams':>14} | {'share of the full table':>23} | {'unseen rate':>11}")
for m in [1, 2, 3, 5]:
    pruned[m] = prune(full_gl, m)
    print(f"{m:>9} | {len(pruned[m]):>14,} | {len(pruned[m]) / len(full_gl):>23.0%} | "
          f"{unseen_rate(pruned[m], test_big_gl):>11.1%}")
```

</details>

In [ ]:
# Stuck on Exercise 3? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 3" above; your own cell stays as it is.)
get_ipython().set_next_input("def prune(counts, min_count):\n    return Counter({g: c for g, c in counts.items() if c >= min_count})\n\n\nfull_gl = count_ngrams(train[\"gl\"], 2)\ntest_big_gl = count_ngrams(test[\"gl\"], 2)\npruned = {}\nprint(f\"{'min count':>9} | {'stored bigrams':>14} | {'share of the full table':>23} | {'unseen rate':>11}\")\nfor m in [1, 2, 3, 5]:\n    pruned[m] = prune(full_gl, m)\n    print(f\"{m:>9} | {len(pruned[m]):>14,} | {len(pruned[m]) / len(full_gl):>23.0%} | \"\n          f\"{unseen_rate(pruned[m], test_big_gl):>11.1%}\")")


**Reflection:** How many test bigrams are unseen with 10,000 training words? With 1.8 million?
How much does a tenfold increase of the training text help? At equal sizes, is Galician
much worse than Spanish? Using the table in Part A, what does a Galician model lose compared
with a Spanish one?

## Part C — A longer n

The lesson's question: is a bigger n better? A longer n-gram is more context, but it is
also rarer. Count the unseen rate for n = 1 to 4 with all the training text.

### ✏️ Exercise 4 — Unseen rate for a longer n

For each n from 1 to 4, count the n-grams of the training text and of the test text,
and compute the unseen rate.

<details><summary><b>Hints</b> (click to open)</summary>

- The training counts come from `train[lang]`, the test counts from `test[lang]`.

</details>

In [ ]:
print(f"{'n':>2} | {'Galician':>8} | {'Spanish':>8}")
by_n = {}
for n in [1, 2, 3, 4]:
    row = []
    for lang in train:
        # TODO: the unseen rate of n-grams of the test text, for this language
        row.append(___)
    by_n[n] = row
    print(f"{n:>2} | {row[0]:>8.1%} | {row[1]:>8.1%}")

In [ ]:
# Check for Exercise 4. Run this cell as is.
assert sorted(by_n) == [1, 2, 3, 4]
assert all(0 <= r <= 1 for row in by_n.values() for r in row)
assert all(by_n[n][k] < by_n[n + 1][k] for n in [1, 2, 3] for k in [0, 1]), "a longer n should be sparser"
assert abs(by_n[2][0] - unseen["gl"][-1]) < 0.005, "n = 2 with all the training text is about the last point of Part B"
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
print(f"{'n':>2} | {'Galician':>8} | {'Spanish':>8}")
by_n = {}
for n in [1, 2, 3, 4]:
    row = []
    for lang in train:
        row.append(unseen_rate(count_ngrams(train[lang], n), count_ngrams(test[lang], n)))
    by_n[n] = row
    print(f"{n:>2} | {row[0]:>8.1%} | {row[1]:>8.1%}")
```

</details>

In [ ]:
# Stuck on Exercise 4? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 4" above; your own cell stays as it is.)
get_ipython().set_next_input("print(f\"{'n':>2} | {'Galician':>8} | {'Spanish':>8}\")\nby_n = {}\nfor n in [1, 2, 3, 4]:\n    row = []\n    for lang in train:\n        row.append(unseen_rate(count_ngrams(train[lang], n), count_ngrams(test[lang], n)))\n    by_n[n] = row\n    print(f\"{n:>2} | {row[0]:>8.1%} | {row[1]:>8.1%}\")")


**Reflection:** How does the unseen rate change from n = 1 to n = 4? What does that mean for a
4-gram model reading new text? Connect it to the lesson's Shakespeare page, where the 4-gram
model often copied Shakespeare word for word.

## Part D — A Galician model writes

A trigram model predicts each word from the two words before it. `generate` starts from two
words and samples each next word in proportion to its count, as the lesson's *Sample*
button did.

### ✏️ Exercise 5 — A trigram model writes

Complete the table `next_words` and the sampling step of `generate`.

- `next_words[(w1, w2)]` is a `Counter` of the words that followed the pair `w1 w2`.
- `generate` picks each next word at random, in proportion to its count:
  `rng.choices(words, weights=counts)` does exactly that.

<details><summary><b>Hints</b> (click to open)</summary>

- The loop variables are `w1, w2, w3` and the count `c`.
- `words, counts = zip(*options.items())` already split the options into two tuples.

</details>

In [ ]:
import random

trigrams_gl = count_ngrams(train["gl"], 3)

next_words = {}
for (w1, w2, w3), c in trigrams_gl.items():
    # TODO: record that the pair (w1, w2) was followed by w3, c times
    ___


def generate(w1, w2, length=25, seed=0):
    rng = random.Random(seed)
    out = [w1, w2]
    for _ in range(length):
        options = next_words.get((out[-2], out[-1]))
        if not options:          # a pair never seen in training: the model is stuck
            out.append("[stuck]")
            break
        words, counts = zip(*options.items())
        # TODO: sample the next word in proportion to its count, and append it
        ___
    return " ".join(out)


for seed in range(3):
    print(generate("a", "miña", seed=seed))
print(generate("unha", "tía"))

In [ ]:
# Check for Exercise 5. Run this cell as is.
def test_table():
    """the table lists every continuation with its count"""
    assert sum(next_words[("a", "miña")].values()) == sum(c for (a, b, _), c in trigrams_gl.items() if (a, b) == ("a", "miña"))

def test_continuations_seen():
    """every generated step is a trigram of the training text"""
    words = generate("a", "miña", seed=1).split()
    words = [w for w in words if w != "[stuck]"]
    assert all((words[i], words[i + 1], words[i + 2]) in trigrams_gl for i in range(len(words) - 2))

def test_proportional():
    """sampling follows the counts"""
    import collections
    picks = collections.Counter(generate("a", "miña", length=1, seed=s).split()[-1] for s in range(400))
    top = next_words[("a", "miña")].most_common(1)[0][0]
    assert picks.most_common(1)[0][0] == top, "the most frequent continuation should be picked most often"

def test_stuck():
    """an unseen pair gets stuck"""
    assert generate("unha", "tía").endswith("[stuck]")

_failed = 0
for _test in [test_table, test_continuations_seen, test_proportional, test_stuck]:
    try:
        _test()
        print("✓", _test.__doc__)
    except Exception as _e:
        _failed += 1
        print("✗", _test.__doc__)
        print("   ", _e if isinstance(_e, AssertionError) else f"{type(_e).__name__}: {_e}")
assert _failed == 0, f"{_failed} test(s) failed: see the ✗ lines above"
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
import random

trigrams_gl = count_ngrams(train["gl"], 3)

next_words = {}
for (w1, w2, w3), c in trigrams_gl.items():
    next_words.setdefault((w1, w2), Counter())[w3] = c


def generate(w1, w2, length=25, seed=0):
    rng = random.Random(seed)
    out = [w1, w2]
    for _ in range(length):
        options = next_words.get((out[-2], out[-1]))
        if not options:          # a pair never seen in training: the model is stuck
            out.append("[stuck]")
            break
        words, counts = zip(*options.items())
        out.append(rng.choices(words, weights=counts)[0])
    return " ".join(out)


for seed in range(3):
    print(generate("a", "miña", seed=seed))
print(generate("unha", "tía"))
```

</details>

In [ ]:
# Stuck on Exercise 5? Run this cell: it puts the solution into a new code cell right below, ready to run.
# (The same code as in the "Solution to Exercise 5" above; your own cell stays as it is.)
get_ipython().set_next_input("import random\n\ntrigrams_gl = count_ngrams(train[\"gl\"], 3)\n\nnext_words = {}\nfor (w1, w2, w3), c in trigrams_gl.items():\n    next_words.setdefault((w1, w2), Counter())[w3] = c\n\n\ndef generate(w1, w2, length=25, seed=0):\n    rng = random.Random(seed)\n    out = [w1, w2]\n    for _ in range(length):\n        options = next_words.get((out[-2], out[-1]))\n        if not options:          # a pair never seen in training: the model is stuck\n            out.append(\"[stuck]\")\n            break\n        words, counts = zip(*options.items())\n        out.append(rng.choices(words, weights=counts)[0])\n    return \" \".join(out)\n\n\nfor seed in range(3):\n    print(generate(\"a\", \"miña\", seed=seed))\nprint(generate(\"unha\", \"tía\"))")


**Reflection:** Do the generated lines look like Galician? Do they make sense beyond a few
words? Why does a line sometimes get stuck? Why does *unha tía* get stuck at once?

## Code literacy check

Before you move on, pick at least two code cells above that you actually ran and read
today, and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.